In [40]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [41]:
df=pd.read_csv("all_kindle_review.csv")
print(df.columns)
df.head()

Index(['Unnamed: 0.1', 'Unnamed: 0', 'asin', 'helpful', 'rating', 'reviewText',
       'reviewTime', 'reviewerID', 'reviewerName', 'summary',
       'unixReviewTime'],
      dtype='str')


,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [42]:
data=df[['reviewText','rating']]
print(f"data shape:{data.shape}")
data.head()

data shape:(12000, 2)


,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [43]:
data.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [44]:
data['rating'].unique()

array([3, 5, 4, 2, 1])

In [45]:
#no imbalance
data['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

DATA PREPROCESSING AND CLEANING

In [46]:
## any rating less than 3 is negative review(0) otherwise positive 1
data['rating']=data['rating'].apply(lambda x: 0 if x<3 else 1)

In [47]:
data['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [48]:
data['reviewText']=data['reviewText'].str.lower()

In [49]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tushi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [50]:
from bs4 import BeautifulSoup

In [51]:
stop_words = set(stopwords.words('english'))

In [52]:
def optimized_clean(text):
    # Convert to string and lowercase
    text = str(text).lower()
    
    # 1. Remove HTML first (before regex, to avoid breaking tags)
    text = BeautifulSoup(text, "html.parser").get_text()
    
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # 3. Remove special characters and numbers (Keep only a-z)
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 4. Remove stopwords and extra whitespace in one go
    # split() handles multiple spaces automatically
    tokens = [word for word in text.split() if word not in stop_words]
    
    return " ".join(tokens)

# Apply ONLY ONCE to the dataframe
print("Starting optimized cleaning...")
data['reviewText'] = data['reviewText'].apply(optimized_clean)
print("Cleaning complete!")

Starting optimized cleaning...
Cleaning complete!


In [53]:
data.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


LEMMATIZATION

In [54]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\tushi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [55]:
lemmatizer=WordNetLemmatizer()

In [56]:
def lemmatize_words(text):
  return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [57]:
data['reviewText']=data['reviewText'].apply(lambda x:lemmatize_words(x))

MODEL TRANING

In [58]:
#train test split
from sklearn.model_selection import train_test_split

In [59]:
x_train,x_test,y_train,y_test=train_test_split(data['reviewText'],data['rating'],test_size=0.20)

In [60]:
print(x_train.shape,x_test.shape)

(9600,) (2400,)


In [62]:
from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer()
x_train_bow=bow.fit_transform(x_train)
x_test_bow=bow.transform(x_test)

In [64]:
print(x_train_bow.shape,x_test_bow.shape)

(9600, 36017) (2400, 36017)


In [65]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
x_train_tfidf=tfidf.fit_transform(x_train)
x_test_tfidf=tfidf.transform(x_test)
print(x_train_tfidf.shape,x_test_tfidf.shape)

(9600, 36017) (2400, 36017)


In [66]:
from sklearn.linear_model import LogisticRegression
lr_model=LogisticRegression(max_iter=1000)
lr_model_bow=lr_model.fit(x_train_bow,y_train)
lr_model_tfidf=lr_model.fit(x_train_tfidf,y_train)

In [67]:
y_pred_bow_lr=lr_model_bow.predict(x_test_bow)
y_pred_tfidf_lr=lr_model_tfidf.predict(x_test_tfidf)

In [68]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report,f1_score

In [69]:
print(f"bow lr acc : :{accuracy_score(y_test,y_pred_bow_lr)}")
print(f"tfidf lr acc : :{accuracy_score(y_test,y_pred_tfidf_lr)}")
print(f"bow lr f1_score : {f1_score(y_test,y_pred_bow_lr)}")
print(f"tfidf lr f1_score : {f1_score(y_test,y_pred_tfidf_lr)}")


bow lr acc : :0.8225
tfidf lr acc : :0.8383333333333334
bow lr f1_score : 0.864244741873805
tfidf lr f1_score : 0.8843172331544424


In [70]:
print("confusion_matrix for bow \n",confusion_matrix(y_test,y_pred_bow_lr))

confusion_matrix for bow 
 [[ 618  191]
 [ 235 1356]]


In [71]:
print("confusion_matrix for tfidf \n",confusion_matrix(y_test,y_pred_tfidf_lr))

confusion_matrix for tfidf 
 [[ 529  280]
 [ 108 1483]]


WITH WORD2VEC

In [72]:
from gensim.models import Word2Vec

In [73]:

import numpy as np

# Word2Vec needs a list of lists (tokenized sentences)
# We use the cleaned 'reviewText' from your previous step
print("Tokenizing sentences...")
sentences = [text.split() for text in data['reviewText']]

print("Training Word2Vec...")
# vector_size=100 is the "sweet spot" for 8GB RAM
w2v_model = Word2Vec(
    sentences=sentences, 
    vector_size=100, 
    window=5, 
    min_count=2, 
    workers=4
)

# Optional: Reduces memory usage by making the model read-only
w2v_model.wv.init_sims(replace=True)

Tokenizing sentences...
Training Word2Vec...


C:\Users\tushi\AppData\Local\Temp\ipykernel_13168\3455912021.py:19: DeprecationWarning: Call to deprecated `init_sims` (Use fill_norms() instead. See https://github.com/RaRe-Technologies/gensim/wiki/Migrating-from-Gensim-3.x-to-4).
  w2v_model.wv.init_sims(replace=True)


In [74]:
def get_vector_sum(tokens, model_wv):
    # Only use words that the model actually learned
    valid_vectors = [model_wv[word] for word in tokens if word in model_wv]
    
    if not valid_vectors:
        return np.zeros(100) # Fallback for empty/unknown reviews
    
    # Summing creates a "weighted" representation of sentiment intensity
    return np.sum(valid_vectors, axis=0)

# Create the Feature Matrix (X) using the word vectors (wv)
X_w2v = np.array([get_vector_sum(s, w2v_model.wv) for s in sentences])
y = data['rating'].values

In [75]:
print(f"Word2Vec feature matrix shape: {X_w2v.shape}")

Word2Vec feature matrix shape: (12000, 100)


In [76]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Split the dense Word2Vec features
x_train_w2v, x_test_w2v, y_train, y_test = train_test_split(X_w2v, y, test_size=0.2, random_state=42)

# Logistic Regression is very fast on your i5, so we can afford to train it on the Word2Vec features
lr_model_2=LogisticRegression(max_iter=1000)
lr_model_w2v=lr_model_2.fit(x_train_w2v, y_train)

y_pred=lr_model_w2v.predict(x_test_w2v)
print(f"Word2Vec + LogReg Accuracy: {accuracy_score(y_test, y_pred)}")


Word2Vec + LogReg Accuracy: 0.7983333333333333


In [77]:
print(f"classification report for w2v + lr \n {classification_report(y_test,y_pred)}")
print(f"confusion matrix for w2v + lr \n {confusion_matrix(y_test,y_pred)}")
print(f"f1 score for w2v + lr : {f1_score(y_test,y_pred)}")

classification report for w2v + lr 
               precision    recall  f1-score   support

           0       0.75      0.60      0.66       803
           1       0.82      0.90      0.86      1597

    accuracy                           0.80      2400
   macro avg       0.78      0.75      0.76      2400
weighted avg       0.79      0.80      0.79      2400

confusion matrix for w2v + lr 
 [[ 480  323]
 [ 161 1436]]
f1 score for w2v + lr : 0.8557806912991657


SAVING THE MODELS AND VECTORIZERS

In [78]:
import joblib

# Save the Vectorizers
joblib.dump(bow, "bow_vectorizer.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

# Save the Classifiers (The ones you trained on BoW, TF-IDF, and W2V)
joblib.dump(lr_model_bow, "model_bow.pkl")
joblib.dump(lr_model_tfidf, "model_tfidf.pkl")
joblib.dump(lr_model_w2v, "model_w2v.pkl")

# Save Word2Vec vectors
w2v_model.wv.save("kindle_vectors.kv")

['model_w2v.pkl']